# Phenomenological weak-field predictions: wide binaries & lensing

This notebook implements the minimal phenomenological mapping from Paper A (SCH) to observable predictions in the weak-field, slow-motion limit. It uses the Poisson reduction:

$$




$$

Assumptions: small coupling (α η << 1), η approximately constant on the system scale, quadratic/torsion terms negligible. See accompanying README for caveats and next steps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi, sqrt

## Key formulas used
From Paper A (phenomenological bridge) we take the modified Poisson equation in the static limit:


- If C_{mu nu} ~ rho eta u_mu u_nu and alpha is small, the potential of a localized mass M with roughly constant eta on the system scale becomes:

  phi(r) = - G M (1 + alpha * eta) / r

- Therefore G_eff = G * (1 + alpha * eta). For small alpha*eta the fractional changes are:
  - delta_v / v = 0.5 * alpha * eta  (circular velocity)
    - delta_P / P = -0.5 * alpha * eta  (orbital period)
  - lensing convergence scales roughly with G_eff, so fractional lensing change ~ alpha * eta

These are leading-order, assumptive mappings intended to give quick intuition and conservative constraints.


In [ ]:
# Constants
G = 6.67430e-11  # m^3 kg^-1 s^-2
M_sun = 1.98847e30  # kg
AU = 1.495978707e11  # m
pc = 3.085677581e16  # m


## Wide binary example
Compute fractional shifts in orbital velocity and period for a 1 Msun - 1 Msun binary at large separation.


In [ ]:
def circular_velocity(M, r, alpha_eta):
    G_eff = G * (1.0 + alpha_eta)
    return np.sqrt(G_eff * M / r)

def orbital_period(M, a, alpha_eta):
    # Keplerian period for reduced-mass ~ M/2 for equal masses but we use simple scaling
    G_eff = G * (1.0 + alpha_eta)
    return 2*pi * np.sqrt(a**3 / (G_eff * M))

# Example system
M = M_sun  # primary mass (kg)
a = 1e4 * AU  # separation ~ 10,000 AU (very wide)

alpha_eta_vals = np.logspace(-8, -1, 60)  # scan alpha*eta
v0 = circular_velocity(M, a, 0.0)
P0 = orbital_period(M, a, 0.0)

dv_over_v = []
dP_over_P = []
for ae in alpha_eta_vals:
    v = circular_velocity(M, a, ae)
    P = orbital_period(M, a, ae)
    dv_over_v.append((v - v0) / v0)
    dP_over_P.append((P - P0) / P0)

dv_over_v = np.array(dv_over_v)
dP_over_P = np.array(dP_over_P)

# quick numeric print for a few reference alpha*eta values
for ae in [1e-4, 1e-5, 1e-6, 1e-7]:
    idx = np.argmin(np.abs(alpha_eta_vals - ae))
    print(f'alpha*eta={ae:.0e}: dv/v={dv_over_v[idx]:.3e}, dP/P={dP_over_P[idx]:.3e}')


## Galaxy lensing example (order-of-magnitude)
Compute fractional lensing signal change for a galaxy treated as a point mass with given stellar mass. For a stacked shear measurement at fixed baryonic mass, fractional change in DeltaSigma ~ alpha*eta to leading order.


In [ ]:
def lensing_fraction_change(alpha_eta):
    # leading-order approximation: fractional change in lensing ~ alpha*eta
    return alpha_eta

M_gal = 1e11 * M_sun  # stellar mass order of magnitude
R = 10 * 3.086e19  # 10 kpc in meters (rough scale)

# compute example fractional shifts
for ae in [1e-2, 1e-3, 1e-4, 1e-5]:
    print(f'alpha*eta={ae:.0e}: fractional lensing change ~ {lensing_fraction_change(ae):.3e}')


## Plots: fractional change vs alpha*eta
Two panels: wide-binary dv/v and fractional lensing change.


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].loglog(alpha_eta_vals, np.abs(dv_over_v), label='|dv/v|')
ax[0].set_xlabel('alpha * eta')
ax[0].set_ylabel('|dv/v|')
ax[0].grid(True)
ax[0].legend()
ax[1].loglog(alpha_eta_vals, alpha_eta_vals, label='fractional lensing ~ alpha*eta')
ax[1].set_xlabel('alpha * eta')
ax[1].set_ylabel('fractional lensing change')
ax[1].grid(True)
ax[1].legend()
plt.suptitle('Phenomenological weak-field examples (SCH)')
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()


## Interpreting these numbers and next steps
- If observational precision on dv/v for a given wide binary sample is, e.g., 10^-3, then the naive bound from that sample is roughly alpha*eta < 2e-3 (since dv/v ~ 0.5 alpha*eta).
- For stacked galaxy lensing with percent-level precision on DeltaSigma, one would constrain alpha*eta < few x 10^-2 (order-of-magnitude) unless eta is large for the systems in the stack.

Next steps to tighten and make rigorous:
1) Compute eta for target systems (wide binaries: are constituent stars expected to have nonzero eta? likely small — need nuclear-scale calculation or lab-calibration).
2) Model spatial variation of eta across a galaxy (C_mu_nu profile).
3) Include screening or quadratic/torsion terms if relevant at the densities considered.
4) For rigorous constraints, perform the Appendix P linearization to map to PPN-like parameters (Option 2).

Outputs: this notebook (phenomenological mapping) and a short README explaining assumptions and how to run it.
